# M38 — Build a Stateful Agent Workflow

**Objective:** build an explicit stateful workflow around model calls and tools.

M37 already handed a validated tool runtime. The useful whole here is a
**stateful workflow**, not another single tool call and not a memory router:

`explicit AgentState → decide → validate → approve → execute → assimilate → terminal`

Chat history is not a state schema. Resume must restore `last_tool_result` and
the ledger snapshot so a completed side effect is not replayed. This teaching
graph is **not a production** agent runtime. RAG, Qdrant, sampling labs, memory
stores, and LangGraph SDKs stay closed.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a `node`, a `terminal`, an `effect_count`, a
`model_turn`, a `last_tool_result` price, or whether an illegal edge mutates state.

Do not hide control flow in a prompt. Do not treat a fluent transcript as
permission to post twice. The repository does not prefill learner answers, ADR
text, or competence.

Canonical sources (named, not imported): `langgraph-docs`, `anthropic-agents`.
Content bundle: `tool-using-agents`.


In [ ]:
from pathlib import Path
import inspect
import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M38" / "agent_workflow.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M37.tool_runtime import CATALOG, RuntimeSession
from missions.M38.agent_workflow import (
    ALLOWED_TRANSITIONS,
    CATALOG_PRICE,
    CATALOG_SKU,
    HANDOFF,
    MAX_STEPS,
    SCALE_LIMIT,
    SEED,
    STATE_FIELDS,
    SYSTEM_MAP,
    WORKFLOW_VERSION,
    InvalidTransition,
    LiveAdapterUnavailable,
    OptionalLangGraphUnavailable,
    apply_transition,
    checkpoint,
    graph_public,
    handoff_contract,
    initial_state,
    observability_report,
    optional_langgraph_compile,
    optional_live_propose,
    pipeline_with_defect,
    repair_run,
    replay_trace,
    resume,
    run_workflow,
)

print("repository root:", ROOT)
print("workflow version:", WORKFLOW_VERSION)
print("seed:", SEED)
print("scale:\n", SCALE_LIMIT)


## M37 → M38 boundary

M37 executed one tool call behind a schema, an approval flag, and an
idempotency key. M38 wraps that runtime in an inspectable state machine.
The model fixture still only *proposes*. Orchestration decides transitions,
loop bounds, checkpoints, and terminals.

Design the **state schema before transitions**. Then run the smallest useful
multi-step task: look up `SKU-7`, then post that catalog price to the cash
ledger after a human approval gate.


In [ ]:
print("state fields:", STATE_FIELDS)
print("graph:")
print(json.dumps(graph_public(), indent=2))
print("allowed from start:", ALLOWED_TRANSITIONS["start"])
print("catalog SKU-7:", CATALOG[CATALOG_SKU])
print("catalog price constant:", CATALOG_PRICE)
print("max_steps:", MAX_STEPS)

try:
    optional_live_propose("lookup SKU-7")
except LiveAdapterUnavailable as exc:
    print("live adapter:", type(exc).__name__)
try:
    optional_langgraph_compile()
except OptionalLangGraphUnavailable as exc:
    print("langgraph adapter:", type(exc).__name__)


The schema lists `last_tool_result`, `completed_effect_keys`, `model_turn`, and
a ledger snapshot *before* any edge is taken. `start` may only go to `decide`.
The catalog price is a fixture fact, not a sampled token.


## Predict before running — whole workflow

Timestamp a prediction before `run-whole`.

Fixed task: look up `SKU-7`, then post that price to the cash ledger. Local
fixtures, not a live model. Approval is granted for this first whole run.

Predict:

- the posted amount
- the terminal node
- whether `effect_count` is 0, 1, or 2
- which two M37 tools execute, in order


In [ ]:
print(SYSTEM_MAP)
whole = run_workflow("purchase_sku7", approval="granted")
print("terminal", whole.state.terminal)
print("node", whole.state.node)
print("effect_count", whole.effect_count)
print("last_tool_result", whole.state.last_tool_result)
print("executions", whole.session.executions)
print("model_turn", whole.state.model_turn)
print("completed keys", whole.state.completed_effect_keys)
print("weights_updated", whole.state.inference["weights_updated"])
print("history dests", [item.dest for item in whole.state.history])
print("handoff\n", HANDOFF)


The map ends at an explicit terminal plus a serializable `AgentState`.
Retrieval packs, vector indexes, and memory routers do not appear. The posted
amount should match the catalog, not a guessed token.


## Predict before running — resume after checkpoint

Timestamp a prediction before `run-resume`.

Change: interrupt after the lookup is assimilated, serialize a checkpoint, then
resume with the same approval policy.

Predict:

- `last_tool_result["price"]` in the checkpoint
- `effect_count` before resume and after resume
- whether `lookup_catalog_price` runs a second time


In [ ]:
first = run_workflow("purchase_sku7", approval="granted", interrupt_when="after_lookup")
print("interrupted", first.interrupted)
print("node", first.state.node)
print("effect_count", first.effect_count)
print("last_tool_result", first.state.last_tool_result)
print("executions", first.session.executions)
payload = checkpoint(first.state, first.session)
print("checkpoint node", payload["node"])
print("checkpoint price", payload["last_tool_result"]["price"])
print("checkpoint json bytes", len(json.dumps(payload)))
resumed = resume(payload, approval="granted")
print("resumed terminal", resumed.state.terminal)
print("resumed effect_count", resumed.effect_count)
print("resumed executions", resumed.session.executions)


A safe checkpoint is a finished node, not mid-execute. Resume should restore
`last_tool_result` so the model fixture can post the assimilated price without
looking the SKU up again. M37 idempotency still applies to the ledger key.


## Predict before running — loop bound

Timestamp a prediction before `run-loop`.

Change: swap only the model fixture to one that keeps requesting an unresolved
lookup. `max_steps=3`. Tools and graph stay fixed.

Predict:

- the terminal
- `model_turn`
- `effect_count` (lookups are not ledger posts)


In [ ]:
looped = run_workflow("unresolved_lookup", max_steps=3, approval="granted")
print("terminal", looped.state.terminal)
print("node", looped.state.node)
print("model_turn", looped.state.model_turn)
print("max_steps", looped.state.max_steps)
print("effect_count", looped.effect_count)
print("executions", looped.session.executions)
print("aborted_ceiling", looped.aborted_ceiling)


Anthropic's agent guidance names a maximum number of iterations as a stopping
condition. That bound lives in orchestration (`model_turn >= max_steps`), not
in a prompt that asks the model to "please stop."


## Predict before running — approval gate

Timestamp a prediction before `run-approval`.

Change: the same SKU-7 post, routed through deny, an interrupt at `approve`,
and a later grant.

Predict:

- `effect_count` after deny
- the node when approval is withheld
- `effect_count` after a granted resume from that interrupt


In [ ]:
denied = run_workflow("purchase_sku7", approval="denied")
print("denied terminal", denied.state.terminal)
print("denied effect_count", denied.effect_count)
print("denied executions", denied.session.executions)
pending = run_workflow("purchase_sku7", approval=None)
print("pending interrupted", pending.interrupted)
print("pending node", pending.state.node)
print("pending effect_count", pending.effect_count)
granted = resume(checkpoint(pending.state, pending.session), approval="granted")
print("granted terminal", granted.state.terminal)
print("granted effect_count", granted.effect_count)


Approval is a node. Deny is a terminal. Withholding approval interrupts without
calling the side-effecting tool. Granting later continues from the checkpoint.


## Predict before running — invalid transition

Timestamp a prediction before `run-invalid`.

Change: from `start`, request `execute` directly. State schema stays fixed.

Predict:

- whether `InvalidTransition` is raised
- whether `state.node` stays `start`
- whether any field in `as_dict()` changes


In [ ]:
invalid_state = initial_state()
before = invalid_state.as_dict()
raised = None
try:
    apply_transition(invalid_state, "execute")
except InvalidTransition as exc:
    raised = exc
print("raised", type(raised).__name__ if raised else None)
print("src", None if raised is None else raised.src)
print("dest", None if raised is None else raised.dest)
print("node after", invalid_state.node)
print("mutated", invalid_state.as_dict() != before)


Illegal edges fail closed. A state machine that "helps" by jumping to execute
is not being generous; it is mutating control flow the schema forbade.


## Predict before running — replay a recorded trace

Timestamp a prediction before `run-replay`.

Change: replay the happy-path proposal list through the deterministic machine.
No live model.

Predict:

- whether the terminal matches `whole`
- whether `effect_count` matches `whole`


In [ ]:
replayed = replay_trace(whole.proposals, approval="granted")
print("original terminal", whole.state.terminal, "effect", whole.effect_count)
print("replay terminal", replayed.state.terminal, "effect", replayed.effect_count)
print("replay executions", replayed.session.executions)
fixture = replay_trace()
print("fixture terminal", fixture.state.terminal, "effect", fixture.effect_count)


Deterministic orchestration can replay a recorded proposal list. The model
fixture is an input. The graph, the bound, and the ledger rules are not.


In [ ]:
labels = ["happy path", "unresolved loop", "denied post"]
terminals = [whole.state.terminal, looped.state.terminal, denied.state.terminal]
rank = {"complete": 3, "loop_exhausted": 2, "denied": 1, "failed": 0, "approve": 0}
heights = [rank.get(item, 0) for item in terminals]
effects = [whole.effect_count, looped.effect_count, denied.effect_count]
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2a9d8f", "#e9c46a", "#e76f51"]
ax.bar(np.arange(len(labels)), heights, color=colors)
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["denied", "loop_exhausted", "complete"])
ax.set_ylabel("terminal rank")
ax.set_xlabel("one named change from the SKU-7 purchase")
ax.set_title("Which run reached complete vs a bound vs deny?")
ax.set_ylim(0, 3.5)
fig.tight_layout()
plt.show()
print("plot asks: which run reached complete vs loop_exhausted vs denied?")
print("terminals", list(zip(labels, terminals)))
print("effect_counts", list(zip(labels, effects)))


## Predict before running — code reading

Timestamp a prediction before `run-code-reading`.

Read `apply_transition`, `run_workflow`, `checkpoint`, `resume`, and
`repair_run` in `missions/M38/agent_workflow.py`. Dump `inspect.getsource`
of those functions and probe live objects (`node`, `last_tool_result`,
`effect_count`, `model_turn`).

Predict:

- whether `apply_transition` mutates `start` when asked for `execute`
- what `run_workflow` stores in `last_tool_result` after lookup
- what `checkpoint` / `resume` restore so the post is not a second lookup
- what `repair_run` reuses from a broken object
- the live `effect_count` after an approved purchase versus a deny


In [ ]:
print(inspect.getsource(apply_transition))
print("--- run_workflow (first lines) ---")
for line in inspect.getsource(run_workflow).splitlines()[:20]:
    print(line)
print("--- checkpoint ---")
print(inspect.getsource(checkpoint))
print("--- resume ---")
print(inspect.getsource(resume))
print("--- repair_run ---")
print(inspect.getsource(repair_run))

print("whole node", whole.state.node)
print("whole effect_count", whole.effect_count)
print("whole last_tool_result amount", whole.state.last_tool_result["amount"])
print("denied effect_count", denied.effect_count)
print("looped model_turn", looped.state.model_turn)
print("looped terminal", looped.state.terminal)
print("resume executions", resumed.session.executions)
print("checkpoint restored price", payload["last_tool_result"]["price"])
illegal = initial_state()
try:
    apply_transition(illegal, "execute")
    illegal_mutated = True
except InvalidTransition:
    illegal_mutated = False
print("illegal start->execute mutated", illegal_mutated, "node", illegal.node)
report = observability_report(whole)
print("handoff", report["handoff"])
print("state fields", report["state_fields"])
print("weights_updated on live state", whole.state.inference["weights_updated"])


## Predict before running — Controlled failure: unbounded loop

Timestamp a prediction before `run-failure`.

An unresolved lookup fixture should hit `loop_exhausted` at `max_steps=3`.
The teaching graph and catalog stay fixed. One named defect removes a gate.

Predict:

- whether `terminal` is `loop_exhausted`
- whether `model_turn` stays at the bound or exceeds it
- whether you should reach for a bigger model first


In [ ]:
broken_loop = pipeline_with_defect(defect="infinite_loop")
print("defect", broken_loop.defect, "claim", broken_loop.claim)
print("terminal", broken_loop.terminal)
print("node", broken_loop.node)
print("model_turn", broken_loop.model_turn)
print("max_steps", broken_loop.audit["max_steps"])
print("aborted_ceiling", broken_loop.audit["aborted_ceiling"])
print("loop_bound_enforced", broken_loop.loop_bound_enforced)
print("effect_count", broken_loop.effect_count)


## Predict before running — Controlled failure: replayed side effect

Timestamp a prediction before `run-failure-replayed`.

A granted SKU-7 post should increment `effect_count` once. One named defect
rewinds orchestration into `execute` without consulting M37 idempotency.

Predict:

- whether `effect_count` is 1 or 2
- whether the two ledger entry ids match
- whether the graph rewind is visible on the broken object


In [ ]:
broken_replayed = pipeline_with_defect(defect="replayed_side_effect")
print("defect", broken_replayed.defect, "claim", broken_replayed.claim)
print("effect_count", broken_replayed.effect_count)
print("first_entry_id", broken_replayed.audit["first_entry_id"])
print("second_entry_id", broken_replayed.audit["second_entry_id"])
print("rewound_from", broken_replayed.audit["rewound_from"])
print("rewound_to", broken_replayed.audit["rewound_to"])
print("idempotency_consulted", broken_replayed.idempotency_consulted)
print("node", broken_replayed.node)


## Diagnose from state, checkpoint, and transition traces

Do not prompt around it. Compare `model_turn` to `max_steps` on the loop
object. Compare `effect_count` and entry ids on the replayed-post object.
Repair orchestration, not the catalog fixture.


## Predict before running — repair the unbounded loop

Timestamp a prediction before `run-failure-repair`.

Call `repair_run` on the **broken loop object only**. Same unresolved policy
and `max_steps`. This cell must not also repair the replayed post.

Predict:

- repaired `terminal`
- repaired `model_turn`
- whether the original broken object still exceeds the bound


In [ ]:
repaired_loop = repair_run(broken_loop)
print(
    "loop repaired",
    repaired_loop.terminal,
    "model_turn",
    repaired_loop.model_turn,
    "bound",
    repaired_loop.loop_bound_enforced,
)
print("broken still", broken_loop.terminal, broken_loop.model_turn)
print("broken still aborted", broken_loop.audit["aborted_ceiling"])
print("infinite-loop repair only; replayed-side-effect repair is a later named change")


The repair must reuse the broken object's initial checkpoint. A second
unrelated happy-path run from module defaults is not a repair. The broken
object remaining unbounded is the regression evidence.


## Predict before running — repair the replayed side effect

Timestamp a prediction before `run-replayed-repair`.

Call `repair_run` on the **broken replayed object only**. Same initial
checkpoint. This cell must not rebind the loop defect.

Predict:

- repaired `effect_count`
- repaired `terminal`
- whether the original broken object still shows two posts


In [ ]:
repaired_replayed = repair_run(broken_replayed)
print(
    "replayed repaired",
    repaired_replayed.terminal,
    "effect",
    repaired_replayed.effect_count,
    "consulted",
    repaired_replayed.idempotency_consulted,
)
print("broken still", broken_replayed.effect_count)
print("broken still consulted", broken_replayed.idempotency_consulted)


Idempotency is composed from M37's ledger key *and* from not re-entering
`execute` after assimilate. Rewinding the node is an orchestration bug.
The broken object remaining at `effect_count == 2` is the regression.


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- SKU-7 lookup-then-post terminal and posted amount
- lookup checkpoint with restored `last_tool_result`
- unresolved-loop `loop_exhausted` at `max_steps=3`
- deny versus grant `effect_count`
- rejected `start -> execute`
- recorded-trace replay
- infinite-loop and replayed-post diagnosis plus `repair_run`

See `missions/M38/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M38/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

Draw a fresh bounded machine for `BIN-4` / qty `9`, name the resume fields,
find a missing terminal, place human approval, and diagnose a replayed
reserve from a state trace.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M38/adr_prompt.md` to choose a V10 workflow state/checkpoint
policy (state schema, persistence boundary, resume, loop limits, approval
points, trace retention). Do not claim a production runtime and do not
implement a memory store, router, or eval harness here.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR. Formal engineering review is required
at M38 for the package, not as a substitute for the learner ADR.


## M37 → M38 → M39 handoff

M37 supplied `run_tool_call` and keyed ledger idempotency. M38 wrapped that
runtime in an explicit state machine with terminals, a loop bound, and
checkpoint/resume.

M39 may add memory, routing, and fallback ladders. It must consume the
serializable state schema and failure/termination semantics. It must not
relabel a missing `last_tool_result` as a prompt problem, and it must not
replay a side effect because chat history "looked" complete.

Reusable artifacts: `AgentState`, `apply_transition`, `run_workflow` /
`resume` / `checkpoint`, typed terminals, and `handoff_contract()`.


## Mission summary

You started with a useful whole: lookup then approved post. You then changed
one variable at a time — resume, loop bound, approval, illegal edge, replay —
and diagnosed two orchestration defects from state traces.

Implementation of this package is not learner completion. Predictions, the
no-AI gate, the ADR, and competence evidence remain yours.


In [ ]:
assert whole.state.terminal == "complete" and whole.effect_count == 1
assert whole.state.last_tool_result["amount"] == CATALOG_PRICE
assert whole.session.executions == ["lookup_catalog_price", "post_ledger_entry"]
assert first.interrupted and first.effect_count == 0
assert first.state.last_tool_result["price"] == CATALOG_PRICE
assert resumed.state.terminal == "complete" and resumed.effect_count == 1
assert resumed.session.executions == ["lookup_catalog_price", "post_ledger_entry"]
assert looped.state.terminal == "loop_exhausted" and looped.state.model_turn == 3
assert denied.state.terminal == "denied" and denied.effect_count == 0
assert pending.interrupted and pending.state.node == "approve"
assert invalid_state.node == "start" and raised is not None
assert replayed.state.terminal == "complete" and replayed.effect_count == 1
assert broken_loop.model_turn > broken_loop.audit["max_steps"]
assert repaired_loop.terminal == "loop_exhausted"
assert broken_loop.terminal != "loop_exhausted"
assert broken_replayed.effect_count == 2 and repaired_replayed.effect_count == 1
assert type(whole.session).__module__ == "missions.M37.tool_runtime"
assert whole.state.inference["weights_updated"] is False
assert isinstance(whole.session, RuntimeSession)
print("M38 integrity checks passed")
print(handoff_contract()["handoff"])
